In [8]:
!rm -rf /content/yolov1-cupy
!git clone https://github.com/mihnea-popescu/yolov1-cupy.git

import sys
sys.path.append('/content/yolov1-cupy')

from main import TestClass

test = TestClass()
test.test()          # IT'S WORKING!

Cloning into 'yolov1-cupy'...
remote: Enumerating objects: 148, done.
remote: Counting objects: 100% (148/148), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 148 (delta 72), reused 107 (delta 39), pack-reused 0 (from 0)
Receiving objects: 100% (148/148), 2.99 MiB | 10.81 MiB/s, done.
Resolving deltas: 100% (72/72), done.
Github classes initialized!


In [9]:
# Pascal VOC 2012 dataset download + extract (Kaggle)
!curl -L -o /content/pascal-voc-2012-dataset.zip https://www.kaggle.com/api/v1/datasets/download/gopalbhattrai/pascal-voc-2012-dataset
print("Extracting Pascal VOC dataset (quiet unzip)...")
!unzip -q /content/pascal-voc-2012-dataset.zip -d /content/yolov1-cupy
!rm /content/pascal-voc-2012-dataset.zip
print("Pascal VOC dataset ready under /content/yolov1-cupy (VOC2012_train_val / VOC2012_test)")

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 3605M  100 3605M    0     0   174M      0  0:00:20  0:00:20 --:--:--  185M
Extracting Pascal VOC dataset (quiet unzip)...
Pascal VOC dataset ready under /content/yolov1-cupy (VOC2012_train_val / VOC2012_test)


In [10]:
# Pascal VOC loader smoke test (after Kaggle download/extract cell)
from pathlib import Path

import cupy as cp
from image_batch_loader import (
    voc_dataset_size,
    voc_num_batches_per_epoch,
    voc_image_target_batch,
)

REPO = "/content/yolov1-cupy"
VOC_DATA_ROOT = "/content/yolov1-cupy/VOC2012_train_val/VOC2012_train_val"

voc_root_candidates = [
    Path("/content/yolov1-cupy/VOC2012_train_val"),
    Path("/content/yolov1-cupy/VOC2012_train_val/VOC2012_train_val"),
    Path("/content/yolov1-cupy/VOC2012_train_val/VOC2012"),
    Path("/content/yolov1-cupy/VOC2012_train_val/VOCdevkit/VOC2012"),
]
print("Checking extracted Pascal VOC paths:")
for candidate in voc_root_candidates:
    print(f"- {candidate}: {'OK' if candidate.exists() else 'missing'}")

train_size = voc_dataset_size(REPO, data_root=VOC_DATA_ROOT, split="train")
val_size = voc_dataset_size(REPO, data_root=VOC_DATA_ROOT, split="val")
train_batches = voc_num_batches_per_epoch(
    REPO,
    batch_size=4,
    data_root=VOC_DATA_ROOT,
    split="train",
)

print(f"VOC train samples: {train_size}")
print(f"VOC val samples: {val_size}")
print(f"VOC train batches @ batch_size=4: {train_batches}")

x_batch, y_batch = voc_image_target_batch(
    REPO,
    batch_size=4,
    seed=0,
    batch_index=0,
    data_root=VOC_DATA_ROOT,
    split="train",
    size=(448, 448),
    s=7,
    b=2,
    c=20,
)

print("x_batch shape:", x_batch.shape, "dtype:", x_batch.dtype)
print("y_batch shape:", y_batch.shape, "dtype:", y_batch.dtype)
print("objects encoded in batch:", int(cp.sum((y_batch[..., 4] > 0) | (y_batch[..., 9] > 0))))

assert x_batch.shape[1:] == (3, 448, 448)
assert y_batch.shape[1:] == (7, 7, 30)
assert y_batch.dtype == cp.float32
print("Pascal VOC loader test passed.")

Checking extracted Pascal VOC paths:
- /content/yolov1-cupy/VOC2012_train_val: OK
- /content/yolov1-cupy/VOC2012_train_val/VOC2012: missing
- /content/yolov1-cupy/VOC2012_train_val/VOCdevkit/VOC2012: missing


FileNotFoundError: Missing VOC split file: /content/yolov1-cupy/VOC2012_train_val/ImageSets/Main/train.txt